# Trabalho A2 — Data Science

**Alunos:** Arthur Ribeiro, Marcos Vitor, Gabriel Asserman e Eduardo Bernardes

## Análise de probabilidade em classificação binária

Objetivo: desenvolver um programa que produza um gráfico baseado em histogramas para 
analisar a distribuição das probabilidades previstas por um modelo de classificação 
binária e apoiar a definição de faixas de decisão.

Modelos de classificação binária normalmente não retornam apenas "0" ou "1" —
eles retornam a probabilidade estimada de uma instância pertencer à classe
positiva. Cabe a quem usa o modelo decidir, a partir dessa probabilidade,
onde ficam os pontos de corte que separam decisões automáticas de casos que
precisam de revisão manual.

**O que esta parte do notebook oferece:**

- `plot_histogram`: gera um histograma comparando a distribuição de
  probabilidades de toda a população com a distribuição apenas da classe
  positiva, já destacando visualmente as três faixas de decisão (negativo
  automático, análise manual, positivo automático).
- `assign_decision_band`: classifica cada instância em uma dessas três
  faixas, a partir de dois cortes `t1 < t2`.
- `decision_band_report`: gera o relatório numérico de cada faixa (percentual
  da população, proporção de positivos/negativos, e as proporções de erro de
  classificação automática, com o denominador de cada uma explícito).

**Reutilização:** as funções trabalham apenas com os rótulos reais (`y_true`)
e as probabilidades estimadas (`y_prob`) — não dependem de nenhum dataset ou
modelo específico. Os cortes `t1`/`t2` devem ser escolhidos observando dados
de **validação** e aplicados ao conjunto de **teste** sem novos ajustes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def assign_decision_band(y_prob, t1, t2):
    """
    Classifica cada instância em uma das 3 faixas de decisão, a partir de dois
    cortes t1 < t2 (definidos previamente com dados de VALIDAÇÃO).

    Retorna um array de strings, uma por instância:
    - "negativo_automatico": probabilidade < t1
    - "analise_manual":      t1 <= probabilidade < t2
    - "positivo_automatico": probabilidade >= t2
    """
    assert t1 < t2, "t1 precisa ser menor que t2"

    y_prob = np.asarray(y_prob)
    faixa = np.full(len(y_prob), "analise_manual", dtype=object)
    faixa[y_prob < t1] = "negativo_automatico"
    faixa[y_prob >= t2] = "positivo_automatico"
    return faixa


def plot_histogram(y_true, y_prob, t1, t2, positive_class_definition, bin_width_pct=10):
    """
    y_true: rótulos reais (0/1) das instâncias.
    y_prob: probabilidade estimada da classe positiva, para TODAS as instâncias
            (independente de y_true) — não filtrar por rótulo real antes de passar aqui.
    t1, t2: cortes (t1 < t2) que definem as 3 faixas de decisão, escolhidos
            previamente com dados de VALIDAÇÃO.
    positive_class_definition: string explicando o que "classe positiva" significa
            neste problema (ex: "é spam?", "é fraude?", "expressa sentimento positivo?").
    bin_width_pct: largura de cada intervalo do histograma, em pontos percentuais
            (ex: 10 -> intervalos de 0-10%, 10-20%, ..., 90-100%).

    Ao gerar os resultados finais do trabalho, chamar esta função apenas com
    y_true/y_prob do CONJUNTO DE TESTE — nunca com dados de treino ou validação.
    """
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    assert len(y_true) == len(y_prob), "cada instância precisa ter rótulo real e probabilidade correspondente"
    assert t1 < t2, "t1 precisa ser menor que t2"
    assert 100 % bin_width_pct == 0, "bin_width_pct precisa dividir 100 igualmente (ex: 5, 10, 20, 25)"

    y_prob_pct = y_prob * 100
    total_instancias = len(y_prob_pct)

    n_bins = 100 // bin_width_pct
    bin_edges = np.linspace(0, 100, n_bins + 1)  # bins meio-abertos: cada instância cai em exatamente um intervalo
    centros = (bin_edges[:-1] + bin_edges[1:]) / 2

    # Mesmos bin_edges para as duas distribuições, para que sejam comparáveis ponto a ponto
    contagem_total, _ = np.histogram(y_prob_pct, bins=bin_edges)
    contagem_positiva, _ = np.histogram(y_prob_pct[y_true == 1], bins=bin_edges)

    # Mesmo denominador (total_instancias) nas duas — o vermelho é um subconjunto do azul,
    # nunca deve ser normalizado pelo próprio total de positivos.
    pct_total = contagem_total / total_instancias * 100
    pct_positiva = contagem_positiva / total_instancias * 100

    t1_pct, t2_pct = t1 * 100, t2 * 100
    faixa = assign_decision_band(y_prob, t1, t2)
    pct_por_faixa = {nome: (faixa == nome).sum() / total_instancias * 100 for nome in
                      ["negativo_automatico", "analise_manual", "positivo_automatico"]}

    largura_barra = bin_width_pct * 0.4

    fig, ax = plt.subplots(figsize=(11, 5.5))

    ax.axvspan(0, t1_pct, color="gray", alpha=0.12)
    ax.axvspan(t1_pct, t2_pct, color="gold", alpha=0.15)
    ax.axvspan(t2_pct, 100, color="green", alpha=0.12)

    ax.bar(centros - largura_barra / 2, pct_total, width=largura_barra,
           color="blue", hatch="//", edgecolor="blue", fill=False, label="Todas as instâncias")
    ax.bar(centros + largura_barra / 2, pct_positiva, width=largura_barra,
           color="red", hatch="//", edgecolor="red", fill=False,
           label=f"Classe positiva ({positive_class_definition})")

    ax.axvline(t1_pct, color="black", linestyle="--", linewidth=1.5)
    ax.axvline(t2_pct, color="black", linestyle="--", linewidth=1.5)

    y_topo = max(pct_total.max(), pct_positiva.max()) * 1.3
    ax.set_ylim(0, y_topo)
    rotulos = {
        "negativo_automatico": ("Negativo automático", (0 + t1_pct) / 2),
        "analise_manual": ("Análise manual", (t1_pct + t2_pct) / 2),
        "positivo_automatico": ("Positivo automático", (t2_pct + 100) / 2),
    }
    for nome, (titulo, x_centro) in rotulos.items():
        ax.text(x_centro, y_topo * 0.97, f"{titulo}\n{pct_por_faixa[nome]:.1f}% da população",
                ha="center", va="top", fontsize=9)

    ax.set_xlabel("Probabilidade estimada da classe positiva (%)")
    ax.set_ylabel("Instâncias do conjunto avaliado (%)")
    ax.set_xlim(0, 100)
    ax.set_xticks(sorted(set(bin_edges) | {t1_pct, t2_pct}))
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
    fig.tight_layout()
    plt.show()

    return fig, ax

In [10]:
import pandas as pd


def decision_band_report(y_true, y_prob, t1, t2):
    """
    Aplica os cortes t1/t2 (já definidos com dados de VALIDAÇÃO) ao conjunto
    passado aqui (deve ser o conjunto de TESTE) e monta o relatório exigido:
    percentual da população por faixa, positivos/negativos por faixa, e as
    duas proporções de erro automatizado, cada uma com seu denominador.
    """
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    faixa = assign_decision_band(y_prob, t1, t2)

    total = len(y_prob)
    total_positivos_reais = (y_true == 1).sum()
    total_negativos_reais = (y_true == 0).sum()

    linhas = []
    for nome_faixa in ["negativo_automatico", "analise_manual", "positivo_automatico"]:
        na_faixa = faixa == nome_faixa
        n = na_faixa.sum()
        pos = ((y_true == 1) & na_faixa).sum()
        neg = ((y_true == 0) & na_faixa).sum()
        linhas.append({
            "Faixa": nome_faixa,
            "Instâncias": n,
            "% da população (n / total_teste)": round(n / total * 100, 2),
            "Positivos reais": pos,
            "% positivos na faixa (pos / n)": round(pos / n * 100, 2) if n else float("nan"),
            "Negativos reais": neg,
            "% negativos na faixa (neg / n)": round(neg / n * 100, 2) if n else float("nan"),
        })
    tabela = pd.DataFrame(linhas)

    # Denominador: total de POSITIVOS REAIS (não o tamanho da faixa, não o total geral)
    positivos_perdidos = ((y_true == 1) & (faixa == "negativo_automatico")).sum()
    prop_positivos_perdidos = positivos_perdidos / total_positivos_reais

    # Denominador: total de NEGATIVOS REAIS (não o tamanho da faixa, não o total geral)
    negativos_mal_classificados = ((y_true == 0) & (faixa == "positivo_automatico")).sum()
    prop_negativos_mal_classificados = negativos_mal_classificados / total_negativos_reais

    print("Relatório por faixa de decisão (conjunto de teste):")
    display(tabela)
    print(
        f"\nPositivos reais classificados automaticamente como negativos: "
        f"{positivos_perdidos}/{total_positivos_reais} "
        f"({prop_positivos_perdidos * 100:.2f}%; denominador = total de positivos reais)"
    )
    print(
        f"Negativos reais classificados automaticamente como positivos: "
        f"{negativos_mal_classificados}/{total_negativos_reais} "
        f"({prop_negativos_mal_classificados * 100:.2f}%; denominador = total de negativos reais)"
    )

    return tabela

# Heart Failure Prediction Dataset

**Fonte:** https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction

**Owner:** fedesoriano

**Licença:** Open Database License (ODbL)

**Descrição:** base de dados clínica com 918 pacientes, combinando registros
de 5 bases distintas (Cleveland, Hungarian, Switzerland, Long Beach VA e
Statlog Heart). Cada linha representa um paciente, descrito por indicadores
clínicos como idade, sexo, tipo de dor no peito, pressão arterial em
repouso, colesterol, glicemia em jejum, resultado do eletrocardiograma em
repouso, frequência cardíaca máxima atingida, angina induzida por exercício
e depressão do segmento ST. O objetivo é prever se o paciente tem ou não
doença cardíaca.

**Variável-Alvo:** `HeartDisease` — **Classe Positiva:** 1 = "o paciente tem doença cardíaca".

In [5]:
import pandas as pd

df = pd.read_csv('./data/heart.csv', sep =',')
df.head(5)

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


Identificamos nossa variável-alvo e não precisaremos fazer nenhuma mudança,
pois o valor já vem formatado em 0 e 1 (binário).

Verificação do percentual de balanceamento.

Isso é muito importante pois existe um grande gap entre bases desbalanceadas
e balanceadas. Bases balanceadas são aquelas que têm valores positivos e
negativos em proporções parecidas — isso mostra que o erro, independente do
lado, tem consequências semelhantes. Bases desbalanceadas, onde temos uma
classe rara, exigem que usemos métricas e tomemos decisões de forma
diferente (como já discutimos: acurácia sozinha enganaria nesse cenário).

In [ ]:
contagem_classes = df.groupby('HeartDisease')['HeartDisease'].count()
porcentagem_classes = contagem_classes / contagem_classes.sum() * 100
print(porcentagem_classes)

A base está bem balanceada (44,7% classe 0 / 55,3% classe 1) — bem diferente
do desbalanceamento extremo visto no spam do professor (86,6%/13,4%). Isso
significa que a acurácia deixa de ser uma métrica enganosa aqui, já que não
existe uma classe "rara" escondendo o desempenho real do modelo.

Porém, balanceamento de classes e custo do erro são coisas diferentes. Mesmo
com a base balanceada, os dois tipos de erro **não têm o mesmo custo** neste
domínio:

- **Falso negativo** (dizer que o paciente não tem doença cardíaca quando na
  verdade tem) é o erro mais grave: o paciente sai sem tratamento, achando
  que está saudável, e o problema pode se agravar sem que ninguém perceba.
- **Falso positivo** (dizer que o paciente tem doença cardíaca quando não
  tem) é um erro mais barato: na pior das hipóteses, o paciente faz alguns
  exames extras para confirmar que está tudo bem.

Por isso, mesmo com a base balanceada, vamos priorizar minimizar falsos
negativos ao escolher os cortes `t1`/`t2` — preferindo encaminhar mais casos
para análise manual (faixa intermediária) a arriscar automatizar uma
classificação negativa que pode estar errada.